# Xarray-spatial
### User Guide: Fire analysis tools
-----

The Fire module has per-cell raster functions for burn severity, fire behavior, and drought indexing. All functions run on four backends (numpy, cupy, dask+numpy, dask+cupy) and take xarray DataArrays as input.

Below, we go through each function using synthetic data.

[dNBR](#dNBR): Differenced Normalized Burn Ratio (pre minus post NBR).

[RdNBR](#RdNBR): Relative dNBR, normalized by pre-fire vegetation density.

[Burn Severity Classification](#Burn-Severity-Classification): USGS 7-class severity from dNBR values.

[Fireline Intensity](#Fireline-Intensity): Byram's fireline intensity (kW/m).

[Flame Length](#Flame-Length): Flame length derived from fireline intensity.

[Rate of Spread](#Rate-of-Spread): Simplified Rothermel model with Anderson 13 fuel types.

[KBDI](#KBDI): Keetch-Byram Drought Index, single time-step update.

-----------

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import xrspatial
from xrspatial.fire import (
    dnbr, rdnbr, burn_severity_class,
    fireline_intensity, flame_length,
    rate_of_spread, kbdi,
)

## Create synthetic fire scenario data

We build a 200x200 landscape with a simulated burn scar in the center. The pre-fire NBR is higher where there is denser vegetation; after the fire, NBR drops inside the burn perimeter. We also create slope, wind, fuel moisture, and weather grids for the fire behavior and drought index functions.

In a real workflow you would compute NBR from satellite imagery using `xrspatial.multispectral.nbr` (see User Guide 6: Remote Sensing).

In [ ]:
H, W = 200, 200
rng = np.random.default_rng(42)

# Coordinates
ys = np.linspace(H - 1, 0, H)
xs = np.linspace(0, W - 1, W)

def make_da(data, name):
    return xr.DataArray(data.astype(np.float32), dims=['y', 'x'],
                        coords={'y': ys, 'x': xs}, name=name)

# Pre-fire NBR: smooth vegetation gradient + noise (range roughly 0.1 to 0.7)
yy, xx = np.meshgrid(np.linspace(0, 1, H), np.linspace(0, 1, W), indexing='ij')
veg_gradient = 0.3 + 0.3 * np.sin(2 * np.pi * yy) * np.cos(np.pi * xx)
pre_nbr = veg_gradient + rng.normal(0, 0.03, (H, W))
pre_nbr = np.clip(pre_nbr, 0.05, 0.85)

# Burn scar: elliptical region in the center where NBR drops
cy, cx = H // 2, W // 2
dist = np.sqrt(((yy - 0.5) / 0.25) ** 2 + ((xx - 0.5) / 0.35) ** 2)
burn_mask = dist < 1.0
burn_intensity = np.clip(1.0 - dist, 0, 1)  # stronger burn near center

post_nbr = pre_nbr.copy()
post_nbr[burn_mask] -= burn_intensity[burn_mask] * (0.3 + rng.uniform(0, 0.3, burn_mask.sum()))
post_nbr = np.clip(post_nbr, -0.5, 0.85)

pre_nbr_agg = make_da(pre_nbr, 'pre_nbr')
post_nbr_agg = make_da(post_nbr, 'post_nbr')

print(f"Pre-fire NBR range: {pre_nbr.min():.3f} to {pre_nbr.max():.3f}")
print(f"Post-fire NBR range: {post_nbr.min():.3f} to {post_nbr.max():.3f}")

veg_cmap = LinearSegmentedColormap.from_list('veg', ['brown', 'yellow', 'green'])

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(pre_nbr_agg.values, cmap=veg_cmap)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side: pre-fire (left) and post-fire (right) NBR
veg_cmap = LinearSegmentedColormap.from_list('veg', ['brown', 'yellow', 'green'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(pre_nbr_agg.values, cmap=veg_cmap)
axes[0].set_title('Pre-fire NBR')
axes[0].set_axis_off()
axes[1].imshow(post_nbr_agg.values, cmap=veg_cmap)
axes[1].set_title('Post-fire NBR')
axes[1].set_axis_off()
plt.tight_layout()
plt.show()

## dNBR

The differenced Normalized Burn Ratio is the simplest burn severity metric: `pre_NBR - post_NBR`. Positive values indicate vegetation loss (burn damage); negative values indicate regrowth or increased greenness. The magnitude roughly corresponds to how much the fire changed the surface.

dNBR is the starting point for most burn severity analyses. USGS and BAER teams use it as input to the severity classification thresholds shown in the next section.

In [ ]:
dnbr_agg = dnbr(pre_nbr_agg, post_nbr_agg)

print(f"dNBR range: {float(dnbr_agg.min()):.3f} to {float(dnbr_agg.max()):.3f}")

fire_cmap = LinearSegmentedColormap.from_list('fire', ['green', 'lightyellow', 'orange', 'red', 'darkred'])

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(dnbr_agg.values, cmap=fire_cmap)
ax.set_axis_off()
plt.tight_layout()
plt.show()

The burn scar is clearly visible in the center. Dark red pixels had the largest drop in NBR (most severely burned), while green areas outside the perimeter show little change.

You can also call this through the `.xrs` accessor:

In [5]:
# Accessor syntax: self = pre-fire NBR, argument = post-fire NBR
dnbr_via_accessor = pre_nbr_agg.xrs.dnbr(post_nbr_agg)
np.testing.assert_allclose(dnbr_via_accessor.data, dnbr_agg.data)
print("Accessor result matches direct call.")

Accessor result matches direct call.


## RdNBR

Relative dNBR normalizes the burn severity by the pre-fire vegetation density:

```
RdNBR = dNBR / sqrt(abs(pre_NBR / 1000))
```

This matters because a dNBR of 0.3 means something different in a dense forest (pre_NBR = 0.7) than in sparse grassland (pre_NBR = 0.2). RdNBR lets you compare severity across vegetation types on the same scale.

Pixels where pre-fire NBR is near zero get set to NaN to avoid dividing by a tiny number.

In [ ]:
rdnbr_agg = rdnbr(dnbr_agg, pre_nbr_agg)

print(f"RdNBR range: {float(np.nanmin(rdnbr_agg.data)):.3f} to "
      f"{float(np.nanmax(rdnbr_agg.data)):.3f}")

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(rdnbr_agg.values, cmap=fire_cmap)
ax.set_axis_off()
plt.tight_layout()
plt.show()

The spatial pattern looks similar to dNBR, but RdNBR rescales by pre-fire vegetation density. Sparse areas that burned can show higher RdNBR even when their raw dNBR was small.

## Burn Severity Classification

The `burn_severity_class` function bins dNBR values into the standard USGS 7-class scheme:

| Class | Label | dNBR range |
|-------|-------|------------|
| 1 | Enhanced regrowth (high) | < -0.251 |
| 2 | Enhanced regrowth (low) | -0.251 to -0.101 |
| 3 | Unburned | -0.101 to 0.099 |
| 4 | Low severity | 0.099 to 0.269 |
| 5 | Moderate-low severity | 0.269 to 0.439 |
| 6 | Moderate-high severity | 0.439 to 0.659 |
| 7 | High severity | >= 0.659 |

Output is int8. Class 0 means nodata (NaN input). This function has `@supports_dataset`, so you can pass an `xr.Dataset` and it will classify each variable.

In [ ]:
severity = burn_severity_class(dnbr_agg)

# Count pixels per class
labels = {
    1: 'Enhanced regrowth (high)',
    2: 'Enhanced regrowth (low)',
    3: 'Unburned',
    4: 'Low severity',
    5: 'Moderate-low',
    6: 'Moderate-high',
    7: 'High severity',
}
for cls, label in labels.items():
    n = int(np.sum(severity.data == cls))
    print(f"  Class {cls} ({label}): {n} pixels")

# Use a discrete colormap: green for unburned/regrowth, yellow-red for severity
severity_float = severity.astype(np.float32)
severity_float.values = np.where(severity_float.values == 0, np.nan, severity_float.values)

severity_cmap = LinearSegmentedColormap.from_list(
    'severity', ['darkgreen', 'green', 'lightgreen', 'yellow', 'orange', 'red', 'darkred'])

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(severity_float.values, cmap=severity_cmap)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Fireline Intensity

Byram's fireline intensity is the rate of heat release per unit length of fire front:

```
I = H * w * R
```

where *H* is the heat content of the fuel (kJ/kg, default 18,000), *w* is fuel consumed per unit area (kg/m²), and *R* is the rate of spread (m/s). The output is in kW/m.

Fireline intensity is the main input for suppression difficulty assessments. Fires below ~350 kW/m can be attacked by hand crews; above ~4,000 kW/m they typically require indirect attack or aerial resources.

We'll create synthetic fuel and spread rate grids to demonstrate.

In [ ]:
# Fuel consumed (kg/m^2): varies with vegetation density
fuel = (veg_gradient * 3.0 + rng.uniform(0, 0.5, (H, W))).astype(np.float32)
fuel_agg = make_da(fuel, 'fuel_consumed')

# Spread rate (m/s): moderate fire, variable across the grid
spread = (0.02 + 0.03 * rng.uniform(0, 1, (H, W))).astype(np.float32)
spread_agg = make_da(spread, 'spread_rate')

intensity_agg = fireline_intensity(fuel_agg, spread_agg, heat_content=18000)

print(f"Fireline intensity range: {float(intensity_agg.min()):.1f} to "
      f"{float(intensity_agg.max()):.1f} kW/m")

ros_cmap = LinearSegmentedColormap.from_list('ros', ['lightyellow', 'orange', 'red', 'darkred'])

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(intensity_agg.values, cmap=ros_cmap)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Flame Length

Flame length in metres, derived from fireline intensity using Byram's equation:

```
L = 0.0775 * I^0.46
```

Negative or zero intensity yields zero flame length. Like `burn_severity_class`, this function supports `@supports_dataset`.

In [ ]:
fl_agg = flame_length(intensity_agg)

print(f"Flame length range: {float(fl_agg.min()):.2f} to {float(fl_agg.max()):.2f} m")

flame_cmap = LinearSegmentedColormap.from_list('flame', ['lightyellow', 'orange', 'red'])

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(fl_agg.values, cmap=flame_cmap)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Rate of Spread

The `rate_of_spread` function implements a simplified Rothermel (1972) spread model. It takes per-pixel slope (degrees), mid-flame wind speed (km/h), and dead fuel moisture (fraction 0-1), plus a scalar fuel model number.

The fuel model selects a row from the Anderson 13 fuel model table, which describes the fuel bed geometry and chemistry (loading, surface-area-to-volume ratio, bed depth, moisture of extinction, etc.). The function pre-computes the Rothermel constants from these parameters and then evaluates the per-pixel spread rate.

### Anderson 13 fuel models at a glance

| Model | Description |
|-------|-------------|
| 1 | Short grass (1 ft) |
| 2 | Timber, grass + understory |
| 3 | Tall grass (2.5 ft) |
| 4 | Chaparral (6 ft) |
| 5 | Brush (2 ft) |
| 6 | Dormant brush / hardwood slash |
| 7 | Southern rough |
| 8 | Closed timber litter |
| 9 | Hardwood litter |
| 10 | Timber, litter + understory |
| 11 | Light logging slash |
| 12 | Medium logging slash |
| 13 | Heavy logging slash |

Below, we vary slope and wind spatially to see how spread rate changes for fuel model 1 (short grass).

In [ ]:
# Create spatially varying inputs
slope_data = (5.0 + 20.0 * yy).astype(np.float32)  # steeper toward the top
wind_data = (5.0 + 15.0 * xx).astype(np.float32)    # windier toward the right
moisture_data = np.full((H, W), 0.06, dtype=np.float32)  # 6% dead fuel moisture

slope_agg = make_da(slope_data, 'slope')
wind_agg = make_da(wind_data, 'wind_speed')
moisture_agg = make_da(moisture_data, 'fuel_moisture')

ros_agg = rate_of_spread(slope_agg, wind_agg, moisture_agg, fuel_model=1)

print(f"Rate of spread range: {float(ros_agg.min()):.2f} to "
      f"{float(ros_agg.max()):.2f} m/min")

ros_cmap = LinearSegmentedColormap.from_list('ros', ['lightyellow', 'orange', 'red', 'darkred'])

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(ros_agg.values, cmap=ros_cmap)
ax.set_axis_off()
plt.tight_layout()
plt.show()

Spread rate is highest in the top-right corner, where slope and wind are both strongest. Fire climbs steep slopes faster, and wind pushes the flame front forward.

### Comparing fuel models

Different fuel types spread fire at very different rates. Below, we compare a few models with the same slope, wind, and moisture.

In [11]:
models_to_compare = [1, 3, 4, 8]
model_names = {1: 'Short grass', 3: 'Tall grass', 4: 'Chaparral', 8: 'Timber litter'}

for fm in models_to_compare:
    r = rate_of_spread(slope_agg, wind_agg, moisture_agg, fuel_model=fm)
    print(f"  Model {fm:2d} ({model_names[fm]:15s}): "
          f"{float(r.min()):8.2f} to {float(r.max()):8.2f} m/min")

  Model  1 (Short grass    ):    50.30 to   705.01 m/min
  Model  3 (Tall grass     ):    34.68 to   519.83 m/min
  Model  4 (Chaparral      ):   109.11 to  1551.39 m/min
  Model  8 (Timber litter  ):     4.46 to    58.96 m/min


### Effect of fuel moisture

Fuel moisture is the biggest factor working against fire spread. As dead fuel moisture approaches the moisture of extinction, reaction intensity drops and spread rate falls off sharply.

In [12]:
# Fixed slope and wind, vary moisture
flat = make_da(np.full((H, W), 10.0, dtype=np.float32), 'slope')
wind10 = make_da(np.full((H, W), 10.0, dtype=np.float32), 'wind')

moistures = [0.03, 0.06, 0.08, 0.10, 0.12]
for m in moistures:
    m_agg = make_da(np.full((H, W), m, dtype=np.float32), 'moisture')
    r = rate_of_spread(flat, wind10, m_agg, fuel_model=1)
    print(f"  Moisture {m:.0%}: {float(r.mean()):8.2f} m/min")

  Moisture 3%:   135.57 m/min
  Moisture 6%:   106.63 m/min
  Moisture 8%:    92.09 m/min
  Moisture 10%:    60.86 m/min
  Moisture 12%:     0.00 m/min


## KBDI

The Keetch-Byram Drought Index tracks cumulative soil moisture deficit (0 to 800 mm). It gets updated daily from maximum temperature and precipitation:

- Precipitation reduces the deficit (after subtracting 5.08 mm for canopy interception).
- On warm days (> 10 °C), evapotranspiration increases the deficit.
- The drought factor depends on the current deficit, temperature, and mean annual precipitation.

Fire weather forecasters use KBDI routinely. Values above 600 mean extreme drought -- organic soils can ignite, and deep-burning fires become hard to suppress.

The `kbdi` function computes a single time-step update. To track KBDI over a season, call it in a loop, feeding each day's output as the next day's `kbdi_prev`.

In [ ]:
# Start from moderate drought (KBDI = 300), hot dry day
kbdi_prev = make_da(np.full((H, W), 300.0, dtype=np.float32), 'kbdi_prev')
max_temp = make_da((30.0 + 5.0 * yy).astype(np.float32), 'max_temp')  # warmer in north
precip = make_da(np.zeros((H, W), dtype=np.float32), 'precip')  # no rain

kbdi_day1 = kbdi(kbdi_prev, max_temp, precip, annual_precip=1200.0)

print(f"KBDI after day 1: {float(kbdi_day1.min()):.1f} to {float(kbdi_day1.max()):.1f}")

drought_cmap = LinearSegmentedColormap.from_list('drought', ['green', 'yellow', 'orange', 'red'])

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(kbdi_day1.values, cmap=drought_cmap)
ax.set_axis_off()
plt.tight_layout()
plt.show()

### Multi-day accumulation

Here we run KBDI forward for 30 hot, dry days, then simulate a 40 mm rain event.

In [14]:
# 30 days of hot, dry weather
current_kbdi = kbdi_prev.copy()
no_rain = make_da(np.zeros((H, W), dtype=np.float32), 'precip')
hot_day = make_da(np.full((H, W), 35.0, dtype=np.float32), 'temp')

history = [float(current_kbdi.mean())]
for day in range(30):
    current_kbdi = kbdi(current_kbdi, hot_day, no_rain, annual_precip=1200.0)
    history.append(float(current_kbdi.mean()))

# Day 31: 40 mm rain event
rain = make_da(np.full((H, W), 40.0, dtype=np.float32), 'precip')
current_kbdi = kbdi(current_kbdi, hot_day, rain, annual_precip=1200.0)
history.append(float(current_kbdi.mean()))

# A few more dry days after the rain
for day in range(5):
    current_kbdi = kbdi(current_kbdi, hot_day, no_rain, annual_precip=1200.0)
    history.append(float(current_kbdi.mean()))

print("Day  0:", f"{history[0]:.1f}")
print("Day 15:", f"{history[15]:.1f}")
print("Day 30:", f"{history[30]:.1f} (pre-rain)")
print("Day 31:", f"{history[31]:.1f} (post-rain)")
print("Day 36:", f"{history[-1]:.1f} (5 days after rain)")

Day  0: 300.0
Day 15: 677.8
Day 30: 770.1 (pre-rain)
Day 31: 741.0 (post-rain)
Day 36: 763.1 (5 days after rain)


The drought index climbs steadily during the dry spell, drops when 40 mm of rain falls (minus 5.08 mm canopy interception), then starts climbing again.

## Putting it together

Post-fire analysis, in short:

1. Compute NBR from pre- and post-fire satellite imagery (`xrspatial.multispectral.nbr`)
2. Compute dNBR and optionally RdNBR
3. Classify burn severity
4. Feed severity maps into rehabilitation planning

For operational fire behavior work:

1. Get slope from a DEM (`xrspatial.slope`)
2. Run `rate_of_spread` with weather forecasts and a fuel model
3. Compute `fireline_intensity` and `flame_length` for suppression planning
4. Track `kbdi` daily for drought conditions

## References

- Key, C.H. and Benson, N.C. (2006). Landscape Assessment (LA). In: Lutes, D.C. et al. (eds), *FIREMON: Fire Effects Monitoring and Inventory System*, USDA Forest Service Gen. Tech. Rep. RMRS-GTR-164-CD.
- Miller, J.D. and Thode, A.E. (2007). Quantifying burn severity in a heterogeneous landscape with a relative version of the delta Normalized Burn Ratio (dNBR). *Remote Sensing of Environment*, 109(1), 66-80.
- Byram, G.M. (1959). Combustion of forest fuels. In: Davis, K.P. (ed), *Forest Fire: Control and Use*, McGraw-Hill.
- Rothermel, R.C. (1972). A mathematical model for predicting fire spread in wildland fuels. USDA Forest Service Res. Pap. INT-115.
- Anderson, H.E. (1982). Aids to determining fuel models for estimating fire behavior. USDA Forest Service Gen. Tech. Rep. INT-122.
- Keetch, J.J. and Byram, G.M. (1968). A drought index for forest fire control. USDA Forest Service Res. Pap. SE-38.
- USGS Burn Severity Portal: https://burnseverity.cr.usgs.gov/